# §E — LoRA-export NLL benchmark (local, three-way)

Quality gate for **NTK Phase-1**: does the `controller_to_lora` exporter preserve what the ntkmirror controller does?

The method under test is `cogflow.transformers.ntkmirror_export.controller_to_lora` — the Phase-1 fine-tune path that turns a trained ntkmirror `SignedLogMaskState` controller into a standard PEFT LoRA adapter (`adapter_config.json` + `adapter_model.safetensors`) that vLLM serves via `--enable-lora`.

This notebook exports the adapter **from the controller, in-notebook** (mirroring the kfp fine-tune component's `weight_lookup` closure exactly — `app/services/ntk_fine_tune_component.py`), then scores three arms on the same held-out GSM8K eval split with the same teacher-forced, completion-masked NLL metric:

1. **floor** — base model, no controller.
2. **reference** — base + ntkmirror controller attached via `ControllerRuntime.apply` forward hooks (the canonical hook-attached number).
3. **lora-export** — the PEFT adapter this notebook just built, merged into the base (the Phase-1 production path).

A wide `floor → reference` gap shows the controller is doing something; a small `reference → lora-export` gap shows the export **preserved** it.

**Gate:** `|lora − reference| / reference ≤ threshold` (plan §E default 3% — the post-residual approximation drops the `(g−1)·h_{i−1}` residual term, so a few % is expected, unlike the §F served gate's 1% band).

This is the notebook counterpart of `evaluate_nll.py`, and it additionally exercises the exporter itself rather than consuming a pre-baked adapter dir. Needs a GPU (or set `DEVICE='cpu'` — slow). Prereqs: `pip install -r ../requirements.txt`.

In [ ]:
# !pip install -r ../requirements.txt   # torch / transformers / peft / datasets / ntkmirror / cogflow

In [ ]:
# --- Config -------------------------------------------------------------
# Paths are relative to the package root (cf_finetune_ntk/), so run the
# notebook from there (Jupyter's default CWD is the notebook's dir; the
# cell below hops up one level if it detects it's running under notebooks/).
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

BASE = "Qwen/Qwen2.5-0.5B-Instruct"        # HF id of the base the controller was trained against
CONTROLLER_PT = "runs/_registered/controller.pt"   # trained SignedLogMaskState .pt (from the kfp run / registered artifact)
ADAPTER_DIR = "runs/_lora_export/adapter"  # where controller_to_lora writes the PEFT adapter this notebook builds

# Eval data: a local JSONL wins if set; otherwise pull the registered
# cogflow eval dataset (same channel the §F served notebook uses).
EVAL_JSONL = "runs/gsm8k_small/eval.jsonl"  # None -> download EVAL_DATASET_ID via cogflow
EVAL_DATASET_ID = "39030a5d-b36c-4f07-8895-667515fcaa14"  # GSM8K test split, 32 held-out rows

DEVICE = "cuda"                             # "cpu" works but is slow on 0.5B
THRESHOLD = 0.03                           # plan §E band: |lora-reference|/reference
# TARGET_MODULES / RANK use controller_to_lora's defaults (o_proj + down_proj,
# rank = max gates-per-layer). Override only to match a bespoke export.
print("cwd:", os.getcwd())

## 1. Load the eval set
The same prompt/completion JSONL the trainer consumes — a local file if `EVAL_JSONL` is set, else pulled from the registered cogflow dataset (unwrap ZIP → find the single `.jsonl`, exactly as the fine-tune component does).

In [ ]:
import json
from pathlib import Path


def load_examples(path):
    """Read the same {prompt, completion} JSONL the trainer consumes."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def download_eval_jsonl(dataset_id, work_dir="runs/_eval"):
    """Pull a cogflow dataset and return the local .jsonl path (unwrap ZIP)."""
    import zipfile
    from cogflow import datasets as cogflow_datasets

    work = Path(work_dir); work.mkdir(parents=True, exist_ok=True)
    downloaded = Path(cogflow_datasets.download_dataset(dataset_id, output_path=str(work)))
    if not zipfile.is_zipfile(downloaded):
        return str(downloaded)
    extracted = work / "extracted"; extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(downloaded) as zf:
        zf.extractall(extracted)
    jsonls = sorted(p for p in extracted.rglob("*") if p.is_file() and p.suffix.lower() == ".jsonl")
    if len(jsonls) != 1:
        raise RuntimeError(f"expected exactly one .jsonl in dataset {dataset_id}, found {jsonls}")
    return str(jsonls[0])


eval_path = EVAL_JSONL or download_eval_jsonl(EVAL_DATASET_ID)
examples = load_examples(eval_path)
print(f"eval set: {eval_path}  ({len(examples)} examples)")

## 2. The NLL metric (shared across all three arms)
Teacher-forced next-token NLL, averaged over the **completion tokens only** (the prompt span is masked). Identical to `evaluate_nll.py::_completion_nll` so the arms are apples-to-apples.

In [ ]:
import math
import torch
import torch.nn.functional as F


def completion_nll(model, tokenizer, examples, device):
    """Teacher-forced NLL over completion tokens only (prompt masked)."""
    model.eval()
    total_logloss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for ex in examples:
            prompt_ids = tokenizer(ex["prompt"], return_tensors="pt").input_ids
            completion_ids = tokenizer(
                ex["completion"], return_tensors="pt", add_special_tokens=False
            ).input_ids
            input_ids = torch.cat([prompt_ids, completion_ids], dim=1).to(device)

            logits = model(input_ids=input_ids).logits  # [1, T, V]
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()

            # Targets at positions >= prompt_len-1 are the completion tokens
            # (one position is lost to the next-token shift).
            prompt_len = prompt_ids.shape[1]
            mask = torch.zeros_like(shift_labels, dtype=torch.bool)
            mask[:, prompt_len - 1:] = True

            losses = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                reduction="none",
            ).view_as(shift_labels)
            selected = losses[mask]
            total_logloss += selected.sum().item()
            total_tokens += selected.numel()
    return total_logloss / max(total_tokens, 1)

## 3. Export the LoRA adapter via `controller_to_lora` (the method under test)

Load the base once, load the trained `SignedLogMaskState`, and build the PEFT adapter with the **exact** `weight_lookup` closure the kfp fine-tune component uses (`o_proj` from `self_attn`, `down_proj` from `mlp`, `.detach()`ed read-only views). The exporter never loads the base itself — it asks the closure for the specific weight matrices it needs per `(layer, module)`.

This is the step that would run inside the pipeline; here we run it locally so the produced adapter is the very thing the `lora-export` arm scores below.

In [ ]:
from ntkmirror import SignedLogMaskState
from transformers import AutoModelForCausalLM, AutoTokenizer
from cogflow.transformers.ntkmirror_export import controller_to_lora

print(f"[export] loading base {BASE!r} to build weight_lookup")
tokenizer = AutoTokenizer.from_pretrained(BASE)
export_model = AutoModelForCausalLM.from_pretrained(BASE).to(DEVICE)

# Fail fast if the base doesn't expose the Llama/Qwen decoder layout the
# closure assumes (same guard as the production component).
if not (
    hasattr(export_model, "model")
    and hasattr(export_model.model, "layers")
    and len(export_model.model.layers) > 0
    and hasattr(export_model.model.layers[0].self_attn, "o_proj")
    and hasattr(export_model.model.layers[0].mlp, "down_proj")
):
    raise RuntimeError(
        f"Base {BASE!r} lacks the expected decoder layout "
        "(model.model.layers[*].self_attn.o_proj + .mlp.down_proj)."
    )


def weight_lookup(layer_i, module_name):
    """Return the base weight matrix for (layer, module). Mirrors
    ntk_fine_tune_component.weight_lookup exactly."""
    decoder_layer = export_model.model.layers[layer_i]
    if module_name == "o_proj":
        return decoder_layer.self_attn.o_proj.weight.detach()
    if module_name == "down_proj":
        return decoder_layer.mlp.down_proj.weight.detach()
    raise ValueError(f"unsupported target module: {module_name}")


state = SignedLogMaskState.load(CONTROLLER_PT)
print(f"[export] controller {CONTROLLER_PT}: {state.n_gates} gates")
adapter_dir = controller_to_lora(
    controller_state=state.to_dict(),
    weight_lookup=weight_lookup,
    output_dir=ADAPTER_DIR,
    base_model_name_or_path=BASE,
)
print(f"[export] LoRA adapter written to {adapter_dir}")
print("[export] files:", sorted(p.name for p in Path(adapter_dir).iterdir()))

# Free the export copy of the base — each arm below loads its own fresh
# base so their states don't interfere (hooks / PEFT wrapping).
del export_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 4. Arm (1) — floor (base only)

In [ ]:
def eval_floor():
    print(f"[floor] loading base {BASE!r}")
    model = AutoModelForCausalLM.from_pretrained(BASE).to(DEVICE)
    nll = completion_nll(model, tokenizer, examples, DEVICE)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"[floor] NLL = {nll:.4f}  ({math.exp(nll):.4f} ppl)")
    return nll


floor_nll = eval_floor()

## 5. Arm (2) — reference (ntkmirror controller via forward hooks)
`ControllerRuntime.apply` installs the signed-log gate as PyTorch forward hooks for the duration of the `with` block and removes them in a `finally` — the canonical hook-attached number the export is trying to reproduce.

In [ ]:
def eval_reference():
    from ntkmirror import ControllerRuntime

    print(f"[reference] loading base {BASE!r} + controller {CONTROLLER_PT}")
    model = AutoModelForCausalLM.from_pretrained(BASE).to(DEVICE)
    ref_state = SignedLogMaskState.load(CONTROLLER_PT)
    runtime = ControllerRuntime(model=model, tokenizer=tokenizer)
    with runtime.apply(ref_state):
        nll = completion_nll(model, tokenizer, examples, DEVICE)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"[reference] NLL = {nll:.4f}  ({math.exp(nll):.4f} ppl)")
    return nll


reference_nll = eval_reference()

## 6. Arm (3) — lora-export (the exported PEFT adapter merged into the base)
The production path: load the base, apply the adapter this notebook exported via PEFT, score. If this tracks the reference, `controller_to_lora` preserved the controller.

In [ ]:
def eval_lora_export():
    from peft import PeftModel

    print(f"[lora-export] loading base {BASE!r} + adapter {ADAPTER_DIR}")
    base_model = AutoModelForCausalLM.from_pretrained(BASE).to(DEVICE)
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).to(DEVICE)
    nll = completion_nll(model, tokenizer, examples, DEVICE)
    del model, base_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"[lora-export] NLL = {nll:.4f}  ({math.exp(nll):.4f} ppl)")
    return nll


lora_nll = eval_lora_export()

## 7. Three-way summary + verdict

In [ ]:
rel = abs(lora_nll - reference_nll) / max(reference_nll, 1e-9)
verdict = "PASS" if rel <= THRESHOLD else "FAIL"

print("================== §E three-way NLL summary ==================")
print(f"floor (base only)              : {floor_nll:.4f}  ({math.exp(floor_nll):.4f} ppl)")
print(
    f"reference (hooks attached)     : {reference_nll:.4f}  "
    f"(Δ vs floor = {reference_nll - floor_nll:+.4f})"
)
print(
    f"production (LoRA-export merged): {lora_nll:.4f}  "
    f"(Δ vs reference = {lora_nll - reference_nll:+.4f}, "
    f"relative = {rel * 100:.2f}%, threshold {THRESHOLD * 100:.0f}%, {verdict})"
)
if reference_nll - floor_nll >= 0:
    print(
        "\n[warn] reference NLL is not below the floor — the controller isn't "
        "improving the eval; the reference→lora comparison may be uninformative."
    )
assert verdict == "PASS", (
    f"LoRA export drifted {rel * 100:.2f}% from the hook-attached reference "
    f"(> {THRESHOLD * 100:.0f}% threshold)"
)
print("\nPASS — the LoRA export preserves the controller within tolerance.")

## Notes

- **What this proves.** A small `reference → lora-export` gap means `controller_to_lora` reproduced the controller's effect through a standard PEFT adapter — so the Phase-1 fine-tune path serves the right thing on vLLM's `--enable-lora`, with no NTK-specific serving infra.
- **Why 3%, not 1%.** The exporter uses the **post-residual approximation** (`exporter.py`): it folds the gate into `o_proj`/`down_proj` rows and drops the `(g−1)·h_{i−1}` residual term, which no static weight edit can express. A few % NLL drift is inherent — that's the approximation error the plan set out to measure. The §F *served* gate is different: it checks a serving pod reproduces the reference exactly (1% band).
- **Same controller everywhere.** All three arms use the one `CONTROLLER_PT`; the reference arm and the exported adapter are built from that exact file, so the comparison is honest.
- **No quantization.** Keep the base in its native dtype for all arms; quantization would add float noise that muddies the approximation signal.
- **Get a controller.** Defaults point at the checked-in `runs/_registered/controller.pt` (Qwen2.5-0.5B GSM8K). To make a fresh one, fine-tune via `POST /cogapi/models/fine-tune` (`export='lora'`) and pull the run's `controller/controller.pt`, or train locally with ntkmirror's `ForwardFineTuner` on `runs/gsm8k_small/train.jsonl`.
- **CPU fallback.** `DEVICE='cpu'` runs but is slow on 0.5B; fine for a smoke test of the export + metric plumbing.